In [465]:
import pathlib
import wave

import numpy as np
import pandas as pd
import xy.pyplot as plt
from scipy.fft import irfft, rfft, rfftfreq
from scipy.io.wavfile import read
from src.note import Note
from scipy.signal import find_peaks
from scipy.ndimage import median_filter

## Preprocessing

In [466]:
DATA_DIR = pathlib.Path("./test_files")

In [467]:
wav_filename = DATA_DIR / "C major.wav"

In [468]:
def load_wav(
    wav_filename: str | pathlib.Path,
) -> tuple[np.ndarray, int]:
    sampling_freq, samples = read(wav_filename)
    with wave.open(str(wav_filename), "rb") as wav:
        bit_depth = wav.getsampwidth() * 8

    samples: np.ndarray = np.asarray(samples, dtype=float)
    if samples.ndim == 1:
        channel = samples
    else:
        channel = samples.sum(axis=1) / samples.shape[1]

    signal: np.ndarray = channel / 2 ** (bit_depth - 1)
    return signal, sampling_freq

In [469]:
signal, sampling_freq = load_wav(wav_filename)

In [470]:
def band_pass_filter(
    signal: np.ndarray,
    sampling_freq: int,
    low_freq: float = 20,
    high_freq: float = 5e3,
) -> np.ndarray:
    assert low_freq < high_freq
    frequencies = rfftfreq(signal.size, d=1 / sampling_freq)
    fourier = rfft(signal)
    fourier[frequencies > high_freq] = 0
    fourier[frequencies < low_freq] = 0
    return irfft(fourier, n=signal.size)

In [471]:
# f_signal = band_pass_filter(
#     signal,
#     sampling_freq,
# )
f_signal = signal

## STFT

In [472]:
WINDOW_SIZE = 8192
HOP_FRACTION = 1 / 2
HOP_SIZE = int(WINDOW_SIZE * HOP_FRACTION)

In [473]:
padded_sig = np.pad(
    signal,
    (0, WINDOW_SIZE - len(signal) % WINDOW_SIZE),
)

In [474]:
def make_segments(
    signal: np.ndarray, window_size=WINDOW_SIZE, hop_size=HOP_SIZE
) -> tuple[list[np.ndarray], list[tuple[int, int]]]:
    starts = np.arange(0, len(signal), hop_size)
    required_length = starts[-1] + window_size
    padded_signal = np.pad(signal, (0, required_length - len(signal)))

    segments = [padded_signal[start : start + window_size] for start in starts]
    indices = [(start, start + window_size) for start in starts]

    return segments, indices

In [475]:
segments, indices = make_segments(f_signal)

In [476]:
stft_df = pd.DataFrame(
    {
        "segment": segments,
        "indices": indices,
    },
)
stft_df["timestamps"] = stft_df["indices"].apply(
    lambda x: (x[0] / sampling_freq, x[1] / sampling_freq)
)

In [477]:
def apply_hann(x: np.ndarray) -> np.ndarray:
    window = np.hanning(len(x))
    return x * window


stft_df["hann_segment"] = stft_df["segment"].apply(apply_hann)

In [478]:
def fourier_segment(x: np.ndarray) -> np.ndarray:
    y = x - x.mean()
    f_seg = rfft(y)
    amplitudes = np.abs(f_seg) / len(y) * 4
    return amplitudes

In [479]:
stft_df["amplitude"] = stft_df["hann_segment"].apply(fourier_segment)

In [480]:
frequencies = rfftfreq(WINDOW_SIZE, 1 / sampling_freq)
frequencies

array([0.00000000e+00, 5.38330078e+00, 1.07666016e+01, ...,
       2.20392334e+04, 2.20446167e+04, 2.20500000e+04], shape=(4097,))

## Spectrogram

In [481]:
spectrogram = np.array(stft_df["amplitude"].to_list())
spectrogram.shape

(160, 4097)

## Note Detection

In [482]:
db_spectrogram = 20 * np.log10(np.clip(spectrogram, a_min=1e-6, a_max=None))

In [483]:
nothing = spectrogram[24]
g2 = spectrogram[140]

In [484]:
thing = g2

### Peak detection

In [485]:
def get_filtersize(
    smoothening_hz: float = 100,
    min_filter_size: int = 5,
    window_size: int = WINDOW_SIZE,
    sampling_freq: float = sampling_freq,
) -> int:
    filtersize = int(smoothening_hz * window_size / sampling_freq)
    filtersize = filtersize if filtersize % 2 else filtersize + 1
    filtersize = max(filtersize, min_filter_size)
    return filtersize


def get_peak_indices(
    signal: np.ndarray,
    filtersize: int,
    signal_noise_ratio: float = 20,
    signal_amplitude_min: float = 1e-5,
) -> list[int]:
    background = median_filter(signal, filtersize)
    peaks, _ = find_peaks(
        signal,
        height=np.clip(
            signal_noise_ratio * background,
            a_min=signal_amplitude_min,
            # a_max=4e-4,
            a_max=None,
        ),
    )
    return peaks

In [486]:
filtersize = get_filtersize(
    100,
)
filtersize

19

In [519]:
spec = spectrogram[95]

In [520]:
peaks = get_peak_indices(spec, filtersize, signal_noise_ratio=20)
fs = frequencies[peaks]
amps = spec[peaks]
plt.plot(frequencies, spec)
plt.plot(
    frequencies,
    np.clip(
        20 * median_filter(spec, filtersize),
        a_min=1e-5,
        a_max=None,
        # a_max=50 * median_filter(spec, filtersize).mean(),
    ),
)
plt.xlim((0, 4e3))
for f in fs:
    plt.axvline(f)
plt.axhline(1e-5)

### Fundamental frequency detection

In [504]:
def detect_fundamentals(
    frequencies: list[float],
    amplitudes: list[float],
    tolerance_cents: float = 50,
    min_harmonics: int = 2,
    max_harmonic: int = 5,
) -> tuple[list[float], list[dict]]:
    fundamentals = []
    stats = []
    remaining = np.ones(len(frequencies), dtype=bool)

    for candidate_index in range(len(frequencies)):
        if not remaining[candidate_index]:
            continue

        seed_frequency = frequencies[candidate_index]
        candidate_frequency = seed_frequency
        available_indices = np.flatnonzero(remaining)
        matched_indices = []
        matched_harmonic_numbers = set()

        for peak_index in available_indices:
            ratio = frequencies[peak_index] / candidate_frequency
            harmonic_number = round(ratio)

            if not 2 <= harmonic_number <= max_harmonic:
                continue

            cents_error = 1200 * abs(np.log2(ratio / harmonic_number))
            if cents_error > tolerance_cents:
                continue

            matched_indices.append(peak_index)
            matched_harmonic_numbers.add(harmonic_number)
            candidate_frequency = frequencies[peak_index] / harmonic_number

        if len(matched_harmonic_numbers) >= min_harmonics:
            fundamental_amplitude = amplitudes[candidate_index]
            loudness_db = 20 * np.log10(max(fundamental_amplitude, np.finfo(float).eps))
            harmonic_score = len(matched_harmonic_numbers)
            matched_indices = np.asarray(matched_indices, dtype=int)

            fundamentals.append(candidate_frequency)
            stats.append(
                {
                    "frequency": candidate_frequency,
                    "harmonics": sorted(matched_harmonic_numbers),
                    "harmonic_score": harmonic_score,
                    "fundamental_amplitude": fundamental_amplitude,
                    "loudness_db": loudness_db,
                }
            )
            # Remove the candidate and all of its observed harmonic peaks.
            remaining[candidate_index] = False
            remaining[matched_indices] = False
    return fundamentals, stats

In [514]:
fundamentals, stats = detect_fundamentals(fs, amps)

### Converting fundamental frequency to note

In [515]:
C0 = Note(NoteNumber=0, Octave=0)
CANDIDATE_NOTES = [(C0 + i) for i in range(12 * 7)]
CANDIDATE_FREQUENCIES = np.array([note.frequency for note in CANDIDATE_NOTES])


def frequencies_to_notes(frequencies: list[float]) -> list[tuple[Note, float]]:
    ans = []
    for freq in frequencies:
        cents_diff = np.abs(1200 * np.log2(CANDIDATE_FREQUENCIES / freq))
        minidx = np.argmin(cents_diff)
        mindiff = np.min(cents_diff)
        ans.append((CANDIDATE_NOTES[minidx], mindiff))
    return ans

In [516]:
mynotes = frequencies_to_notes(fundamentals)
mynotes, [n[0].name for n in mynotes]

([(Note(NoteNumber=4, Octave=4, sharp=True), np.float64(6.5576087554337485))],
 ['E4'])

In [522]:
peak_rows = []
note_rows = []

frame_times = np.array([np.mean(index) / sampling_freq for index in indices])
frequency_limit = 5e3

for frame_index, spectrum in enumerate(spectrogram):
    peaks = get_peak_indices(
        spectrum,
        filtersize,
    )

    peak_rows.extend(
        {
            "frame_index": frame_index,
            "time": frame_times[frame_index],
            "frequency": frequencies[peak_index],
            "amplitude": spectrum[peak_index],
        }
        for peak_index in peaks
    )

    frame_frequencies = frequencies[peaks]
    frame_amplitudes = spectrum[peaks]
    frame_fundamentals, frame_stats = detect_fundamentals(
        frame_frequencies,
        frame_amplitudes,
    )
    frame_notes = frequencies_to_notes(frame_fundamentals)

    for fundamental, fundamental_stats, (note, cents_error) in zip(
        frame_fundamentals,
        frame_stats,
        frame_notes,
    ):
        note_rows.append(
            {
                "frame_index": frame_index,
                "time": frame_times[frame_index],
                "frequency": fundamental,
                "note": note.name,
                "cents_error": cents_error,
                "harmonics": fundamental_stats["harmonics"],
                "harmonic_score": fundamental_stats["harmonic_score"],
                "fundamental_amplitude": fundamental_stats["fundamental_amplitude"],
                "loudness_db": fundamental_stats["loudness_db"],
            }
        )

all_peaks = pd.DataFrame(peak_rows)
detected_notes = pd.DataFrame(note_rows)
frequency_mask = frequencies <= frequency_limit
spectrogram_for_plot = spectrogram[:, frequency_mask]
spectrogram_db = 20 * np.log10(np.maximum(spectrogram_for_plot, np.finfo(float).eps))
spectrogram_db -= spectrogram_db.max()

fig, ax = plt.subplots(figsize=(20, 10))
image = ax.pcolormesh(
    np.arange(len(spectrogram)),
    frequencies[frequency_mask],
    spectrogram_db.T,
    shading="auto",
    cmap="magma",
    vmin=-80,
)

visible_peaks = all_peaks[all_peaks["frequency"] <= frequency_limit]
ax.scatter(
    visible_peaks["frame_index"],
    visible_peaks["frequency"],
    s=12,
    color="cyan",
    linewidths=0.8,
    label="Spectral peaks",
)

visible_notes = detected_notes[detected_notes["frequency"] <= frequency_limit]
ax.scatter(
    visible_notes["frame_index"],
    visible_notes["frequency"],
    s=45,
    color="lime",
    marker="x",
    linewidths=1.2,
    label="Detected fundamentals",
)

label_notes = visible_notes.drop_duplicates(["frame_index", "note"])
label_starts = label_notes.groupby("note")["frame_index"].diff().ne(1)
for detection in label_notes[label_starts].itertuples(index=False):
    ax.annotate(
        detection.note,
        (detection.frame_index, detection.frequency),
        xytext=(4, 6),
        textcoords="offset points",
        fontsize=8,
        color="white",
    )

ax.set_title("Spectral Peaks and Detected Notes")
ax.set_xlabel("Frame index")
ax.set_ylabel("Frequency (Hz)")
ax.set_ylim(0, frequency_limit)
ax.legend()
fig.colorbar(image, ax=ax, label="Magnitude (dB)")
plt.show()

## Causal note detector

In [501]:
N_PREV_FRAMES = 5
print(
    f"Time to detect : {(WINDOW_SIZE + (N_PREV_FRAMES - 1) * HOP_SIZE) / sampling_freq}"
)

Time to detect : 0.5572789115646258
